# 🎨 NÉT STUDIO — HUẤN LUYỆN AI MÓC KHOÁ ĐỘC BẢN (LoRA + SDXL)
Notebook này được thiết kế để chạy **100% MIỄN PHÍ** trên Google Colab (GPU T4).

### Kết quả đạt được:
* Xuất ra file trọng số: `net_keychain_sdxl_lora.safetensors` (~70MB).
* Khi kết hợp với Prompt hoặc IP-Adapter, AI sẽ luôn tạo ra **móc khoá 2D chuẩn xưởng (có khoen xỏ kim loại trên đỉnh, viền die-cut bo cong, nền trắng tinh khiết)**.

### Bước 1: Kiểm tra GPU & Cài đặt môi trường cần thiết

In [ ]:
!nvidia-smi

# Cài đặt thư viện huấn luyện khuếch tán mới nhất từ Hugging Face & torchao
!pip install -q diffusers transformers accelerate peft bitsandbytes xformers torchvision safetensors --upgrade torchao

### Bước 2: Chuẩn bị Dataset (Chọn Cách 2A nếu chưa có ảnh, hoặc Cách 2B nếu tự tải ảnh lên)

In [ ]:
# =========================================================================
# CÁCH 2A: TỰ ĐỘNG SINH 15 ẢNH MẪU CHUẨN XƯỞNG (Nếu bạn chưa có sẵn ảnh mẫu)
# Chạy ô này mất khoảng 2 phút trên GPU T4, tự tạo 15 ảnh mẫu + file metadata.jsonl
# =========================================================================
import os
import json
import torch
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler

os.makedirs("dataset", exist_ok=True)
os.makedirs("output_lora", exist_ok=True)

sample_prompts = [
    "net_keychain, a 2D flat die-cut acrylic keychain charm of a cool chibi Vietnamese boy with stylish glasses, black t-shirt, direct front view, bold clean black contour die-cut outline, vibrant flat colors, top hanging loop with attached silver metal key ring, isolated on pure solid white #FFFFFF background, no shadows",
    "net_keychain, a 2D flat die-cut acrylic keychain charm of an energetic chibi girl with short brown hair, yellow bucket hat, pastel blue hoodie, direct front view, bold clean black contour die-cut outline, vibrant flat colors, top hanging loop with attached silver metal key ring, isolated on pure solid white #FFFFFF background, no shadows",
    "net_keychain, a 2D flat die-cut acrylic keychain charm of an adorable chubby baby smiling with rosy cheeks, wearing a knitted frog beanie, direct front view, bold clean black contour die-cut outline, vibrant flat colors, top hanging loop with attached silver metal key ring, isolated on pure solid white #FFFFFF background, no shadows",
    "net_keychain, a 2D flat die-cut acrylic keychain charm of a cute tourist couple standing together, boy with green cap, girl with floral sunglasses, direct front view, bold clean black contour die-cut outline, vibrant flat colors, top hanging loop with attached silver metal key ring, isolated on pure solid white #FFFFFF background, no shadows",
    "net_keychain, a 2D flat die-cut acrylic keychain charm of a young barista holding an iced milk coffee cup, direct front view, bold clean black contour die-cut outline, vibrant flat colors, top hanging loop with attached silver metal key ring, isolated on pure solid white #FFFFFF background, no shadows",
    "net_keychain, a 2D flat die-cut acrylic keychain charm of an adorable fluffy brown poodle puppy with big glossy eyes, tiny red collar, direct front view, bold clean black contour die-cut outline, vibrant flat colors, top hanging loop with attached silver metal key ring, isolated on pure solid white #FFFFFF background, no shadows",
    "net_keychain, a 2D flat die-cut acrylic keychain charm of a chubby orange tabby cat sitting politely, wearing an ocean wave scarf, direct front view, bold clean black contour die-cut outline, vibrant flat colors, top hanging loop with attached silver metal key ring, isolated on pure solid white #FFFFFF background, no shadows",
    "net_keychain, a 2D flat die-cut acrylic keychain charm of an endangered Son Tra red-shanked douc langur monkey holding a tropical leaf, direct front view, bold clean black contour die-cut outline, vibrant flat colors, top hanging loop with attached silver metal key ring, isolated on pure solid white #FFFFFF background, no shadows",
    "net_keychain, a 2D flat die-cut acrylic keychain charm of an ocean sea turtle with turquoise shell swimming happily, direct front view, bold clean black contour die-cut outline, vibrant flat colors, top hanging loop with attached silver metal key ring, isolated on pure solid white #FFFFFF background, no shadows",
    "net_keychain, a 2D flat die-cut acrylic keychain charm of a blooming pink lotus flower with emerald green pod, authentic minimalist flora, direct front view, bold clean black contour die-cut outline, vibrant flat colors, top hanging loop with attached silver metal key ring, isolated on pure solid white #FFFFFF background, no shadows",
    "net_keychain, a 2D flat die-cut acrylic keychain charm of Da Nang Dragon Bridge breathing fire, stylized waves, direct front view, bold clean black contour die-cut outline, vibrant flat colors, top hanging loop with attached silver metal key ring, isolated on pure solid white #FFFFFF background, no shadows",
    "net_keychain, a 2D flat die-cut acrylic keychain charm of a traditional Vietnamese aluminum coffee phin dropping coffee, direct front view, bold clean black contour die-cut outline, vibrant flat colors, top hanging loop with attached silver metal key ring, isolated on pure solid white #FFFFFF background, no shadows"
]

print("⏳ Đang tải SDXL để tạo 12 ảnh mẫu chuẩn xưởng...")
pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

records = []
for idx, p in enumerate(sample_prompts):
    fname = f"charm_{idx+1:02d}.png"
    fpath = os.path.join("dataset", fname)
    print(f"[{idx+1}/{len(sample_prompts)}] Đang vẽ ảnh mẫu: {fname}...")
    img = pipe(prompt=p, num_inference_steps=20, guidance_scale=7.5).images[0]
    img.save(fpath)
    records.append({"file_name": fname, "text": p})

# Ghi file metadata.jsonl bắt buộc cho Diffusers LoRA
with open("dataset/metadata.jsonl", "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

del pipe
torch.cuda.empty_cache()
print("✓ THÀNH CÔNG! Đã khởi tạo xong 12 ảnh mẫu móc khoá và file dataset/metadata.jsonl!")

### Bước 3: Tải script huấn luyện chuẩn Diffusers SDXL LoRA

In [ ]:
!wget -q -O train_lora_sdxl.py https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora_sdxl.py
print('✓ Đã tải script train_lora_sdxl.py thành công!')

### Bước 4: Khởi động quá trình Huấn Luyện (Training Loop ~45-60 phút)

In [ ]:
!pip install -q --upgrade torchao
!sed -i 's/check_min_version(".*")/pass/g' train_lora_sdxl.py

!accelerate launch \
  --mixed_precision="fp16" \
  --num_processes=1 \
  --num_machines=1 \
  --dynamo_backend="no" \
  train_lora_sdxl.py \
  --mixed_precision="fp16" \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --train_data_dir="./dataset" \
  --resolution=1024 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --gradient_checkpointing \
  --use_8bit_adam \
  --max_train_steps=800 \
  --learning_rate=1e-4 \
  --lr_scheduler="cosine" \
  --lr_warmup_steps=80 \
  --rank=16 \
  --checkpointing_steps=200 \
  --checkpoints_total_limit=2 \
  --output_dir="./output_lora" \
  --seed=42

### Bước 5: Kiểm tra kết quả (Test Inference ngay trong Colab)

In [ ]:
import torch
from diffusers import DiffusionPipeline

pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

# Nạp trọng số LoRA Nét Keychain vừa huấn luyện
pipe.load_lora_weights("./output_lora")

test_prompt = "net_keychain, a 2D flat die-cut acrylic keychain charm of a cool chibi Vietnamese boy with stylish glasses, black t-shirt, direct front view, bold clean black contour die-cut outline, vibrant flat colors, top hanging loop with attached silver metal key ring, isolated on pure solid white #FFFFFF background, no shadows"

image = pipe(prompt=test_prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
image.save("test_net_keychain.png")
display(image)

### Bước 6: Tải file Model `net_keychain_lora.safetensors` về máy tính

In [ ]:
from google.colab import files
import glob

lora_files = glob.glob("./output_lora/*.safetensors")
if lora_files:
    print(f'Đang tải file {lora_files[0]} về máy tính...')
    files.download(lora_files[0])
else:
    print('Chưa tìm thấy file .safetensors. Vui lòng kiểm tra lại bước huấn luyện!')